# CNN Model Training

This notebook defines the CNN architecture using Keras Sequential API,
trains with EarlyStopping & ModelCheckpoint callbacks, saves the best weights,
and persists the training history for later evaluation.

In [ ]:
import sys
import pickle
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))
from src.pipeline.data_pipeline import get_train_val_test
from src.pipeline.model import build_cnn_model, compile_model, train_model

BATCH_SIZE = 32
IMG_SIZE = (150, 150)
NUM_CLASSES = len([
    p for p in Path("../data/processed/train").iterdir() if p.is_dir()
])
print(f"Detected {NUM_CLASSES} classes")

# Load data
train_ds, val_ds, test_ds = get_train_val_test(
    data_dir="../data/processed",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

# Build and compile model
model = build_cnn_model(
    input_shape=(*IMG_SIZE, 3),
    num_classes=NUM_CLASSES,
    dropout_rate=0.5,
    dense_units=128,
)
compile_model(model, learning_rate=0.001)
model.summary()

In [ ]:
EPOCHS = 50
CHECKPOINT_DIR = "../models_output/checkpoints"
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

# Train model — EarlyStopping will prevent overfitting,
# ModelCheckpoint saves the best weights to best.keras
history = train_model(
    model,
    train_ds,
    val_ds,
    epochs=EPOCHS,
    checkpoint_path=f"{CHECKPOINT_DIR}/best.keras",
)

# Persist history for 04_evaluation.ipynb
hist_path = Path("../reports/figures/training_history.pkl")
hist_path.parent.mkdir(parents=True, exist_ok=True)
with open(hist_path, "wb") as f:
    pickle.dump(history.history, f)
print(f"Training history saved to {hist_path}")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
ax1.set_title('Accuracy over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Validation Loss')
ax2.set_title('Loss over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig("../reports/figures/accuracy_loss_curves.png", dpi=150)
plt.show()
print("Saved training curves to reports/figures/accuracy_loss_curves.png")

# Print final metrics
best_epoch = len(history.history['val_accuracy']) - 1
print(f"\nFinal — Train acc: {history.history['accuracy'][-1]:.4f}, Val acc: {history.history['val_accuracy'][-1]:.4f}")